# Three Sources of Endogeneity

**Course**: Quantitative Econometrics  
**Audience**: Bachelor students  
**Instructor**: Swapnil Singh

This notebook is still **pre-IV**. We are learning the problem first. At the end of each section, we give a short preview of how an external source of variation could fix the problem, but the formal IV lecture comes later.

## Learning goals

1. See three different ways to create the same problem: $\operatorname{cov}(X_i, u_i) \neq 0$
2. Connect intuitive stories, algebra, and simulation output
3. Learn which settings can be fixed with controls and which ones require something stronger

## One framework, three mechanisms

We keep returning to the same regression:
$$
Y_i = \beta_0 + \beta_1 X_i + u_i.
$$

OLS needs $X_i$ to be unrelated to the hidden term $u_i$. The three mechanisms below break that requirement in different ways:

| Source | What goes wrong? | Why does $X$ become correlated with $u$? |
|---|---|---|
| Omitted variable | Something relevant is left out | The omitted factor affects both $X$ and $Y$ |
| Measurement error | We observe a noisy version of $X$ | The noise contaminates both the regressor and the error |
| Simultaneity | $X$ and $Y$ are jointly chosen | The regressor already responds to shocks in the outcome equation |

## Setup

Run the next cell once at the start of the presentation. If the kernel restarts during RISE, come back and run it again.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

candidate_dirs = [Path.cwd(), Path.cwd() / "lectures" / "code"]
for candidate in candidate_dirs:
    if (candidate / "endogeneity_lab.py").exists():
        sys.path.insert(0, str(candidate))
        break

import endogeneity_lab as lab

lab.set_plot_style()
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [ ]:
sample_size = 400
mc_reps = 700

## 1. Omitted Variable Bias

### Intuition

Suppose we want to estimate the effect of schooling on earnings, but we omit ability.

- ability affects earnings directly
- ability also affects schooling

Then ability is hidden inside $u$, and the regressor `X` inherits part of it.

### Algebra

True model:
$$
Y_i = \beta_0 + \beta_1 X_i + \beta_2 W_i + \varepsilon_i
$$

Estimated short regression:
$$
Y_i = \beta_0 + \beta_1 X_i + u_i, \quad u_i = \beta_2 W_i + \varepsilon_i
$$

Therefore,
$$
\operatorname{cov}(X_i, u_i) = \beta_2 \operatorname{cov}(X_i, W_i).
$$

Omitted variable bias disappears if either:

- the omitted factor does not matter for $Y$ (`beta_w = 0`)
- the omitted factor is unrelated to $X$ (`rho_xw = 0`)

In [ ]:
if "lab" not in globals():
    raise RuntimeError("Run the setup cell on the previous slide first.")

ovb_params = {
    "n": sample_size,
    "beta_x": 1.0,
    "beta_w": 1.5,
    "rho_xw": 0.8,
    "instrument_strength": 0.9,
    "x_noise_sd": 1.0,
}

In [ ]:
if "lab" not in globals():
    raise RuntimeError("Run the setup cell on the previous slide first.")

ovb_data, ovb_snapshot = lab.ovb_one_run(**ovb_params, seed=2026)

display(ovb_snapshot)
lab.plot_ovb_sample(ovb_data, beta_x=ovb_params["beta_x"])

Read the snapshot like this:

- `OLS omit W` is the biased short regression
- `OLS control for W` is the clean benchmark when the omitted variable becomes observable
- `IV using Z` is only a preview for later: if we had a clean shifter `Z`, it could also isolate variation in `X` that is unrelated to `W`

If you want to turn off omitted-variable bias and verify the theory, set `rho_xw = 0` or `beta_w = 0` and rerun the two cells above.

In [ ]:
if "lab" not in globals():
    raise RuntimeError("Run the setup cell on the previous slide first.")

ovb_mc = lab.mc_ovb(**ovb_params, reps=mc_reps, seed=1)
ovb_summary = lab.summarize_estimates(ovb_mc, truth=ovb_params["beta_x"])

display(ovb_summary)
lab.plot_estimate_distributions(
    ovb_mc,
    truth=ovb_params["beta_x"],
    title="Omitted variable bias: the short regression drifts away from the truth",
    x_label="Estimated slope on X",
)

## 2. Measurement Error

### Intuition

Sometimes the regressor is conceptually right, but measured badly. Think of self-reported income, hours worked, or wealth.

Let the true regressor be $X_i^\*$, but suppose we observe:
$$
X_i = X_i^\* + w_i.
$$

### Algebra

If the true model is
$$
Y_i = \beta_0 + \beta_1 X_i^\* + \varepsilon_i,
$$
then after substituting $X_i^\* = X_i - w_i$ we get
$$
Y_i = \beta_0 + \beta_1 X_i + (\varepsilon_i - \beta_1 w_i).
$$

The new error term contains the measurement noise. So the observed regressor and the error term now overlap mechanically.

Under classical measurement error, OLS is biased toward zero:
$$
\hat{\beta}_1^{OLS} \to \beta_1 \frac{\operatorname{var}(X^\*)}{\operatorname{var}(X^\*) + \operatorname{var}(w)}.
$$

In [ ]:
if "lab" not in globals():
    raise RuntimeError("Run the setup cell on the previous slide first.")

measurement_params = {
    "n": sample_size,
    "beta_x": 1.0,
    "instrument_strength": 0.9,
    "latent_noise_sd": 1.0,
    "measurement_noise_sd": 1.1,
}

In [ ]:
if "lab" not in globals():
    raise RuntimeError("Run the setup cell on the previous slide first.")

me_data, me_snapshot = lab.measurement_error_one_run(**measurement_params, seed=2026)

display(me_snapshot)
lab.plot_measurement_error_sample(me_data, beta_x=measurement_params["beta_x"])

Focus on the slope estimates:

- `OLS using true X*` is the benchmark we would like to have
- `OLS using noisy X` is flatter because measurement noise pushes the coefficient toward zero
- `IV using Z` is a preview of how an external source of clean variation could recover the true signal

To see attenuation bias disappear, set `measurement_noise_sd = 0` and rerun the last two cells.

In [ ]:
if "lab" not in globals():
    raise RuntimeError("Run the setup cell on the previous slide first.")

me_mc = lab.mc_measurement_error(**measurement_params, reps=mc_reps, seed=2)
me_summary = lab.summarize_estimates(me_mc, truth=measurement_params["beta_x"])

display(me_summary)
lab.plot_estimate_distributions(
    me_mc,
    truth=measurement_params["beta_x"],
    title="Measurement error: OLS is pulled toward zero",
    x_label="Estimated slope on X",
)

## 3. Simultaneity

### Intuition

In supply and demand, price and quantity are determined together. If we run a regression of quantity on price, price is not an outside cause that arrives first. It is an equilibrium outcome.

Demand:
$$
Q_i = \alpha_0 + \alpha_1 P_i + u_i^d
$$

Supply:
$$
Q_i = \gamma_0 + \gamma_1 P_i + \delta Z_i + u_i^s
$$

Here `Z` is a supply shifter. We use it only as a preview device for later.

In [ ]:
if "lab" not in globals():
    raise RuntimeError("Run the setup cell on the previous slide first.")

lab.plot_supply_demand_worlds(seed=2026)

The figure above is the key intuition:

- if both supply and demand move, the cloud of equilibrium points does not trace a single structural curve
- if only supply shifts, the equilibrium points trace out demand
- if only demand shifts, the equilibrium points trace out supply

This is why simultaneity is not just another omitted-variable story. The regressor itself is jointly determined inside the system.

In [ ]:
if "lab" not in globals():
    raise RuntimeError("Run the setup cell on the previous slide first.")

simultaneity_params = {
    "n": sample_size,
    "demand_intercept": 10.0,
    "demand_slope": -0.8,
    "supply_intercept": 2.0,
    "supply_slope": 1.2,
    "instrument_strength": 1.5,
    "demand_shock_sd": 1.0,
    "supply_shock_sd": 1.0,
}

In [ ]:
if "lab" not in globals():
    raise RuntimeError("Run the setup cell on the previous slide first.")

sim_data, sim_snapshot = lab.simultaneity_one_run(**simultaneity_params, seed=2026)

display(sim_snapshot)

The important numbers are:

- `corr(P, demand shock)`: this shows price is endogenous in the demand equation
- `OLS of Q on P`: this does not recover the true demand slope
- `IV using supply shifter Z`: preview of the idea that a variable shifting supply but not demand can identify the demand curve

In [ ]:
if "lab" not in globals():
    raise RuntimeError("Run the setup cell on the previous slide first.")

sim_mc = lab.mc_simultaneity(**simultaneity_params, reps=mc_reps, seed=3)
sim_summary = lab.summarize_estimates(sim_mc, truth=simultaneity_params["demand_slope"])

display(sim_summary)
lab.plot_estimate_distributions(
    sim_mc,
    truth=simultaneity_params["demand_slope"],
    title="Simultaneity: OLS mixes supply and demand",
    x_label="Estimated demand slope",
    xlim=(-1.6, 1.2),
)

## Side-by-side comparison

The three cases look different economically, but from the point of view of regression they all create the same symptom: `X` carries hidden information from the error term.

In [ ]:
if "lab" not in globals():
    raise RuntimeError("Run the setup cell on the previous slide first.")

if "mc_reps" not in globals():
    mc_reps = 700

if "ovb_params" not in globals():
    ovb_params = {
        "n": 400,
        "beta_x": 1.0,
        "beta_w": 1.5,
        "rho_xw": 0.8,
        "instrument_strength": 0.9,
        "x_noise_sd": 1.0,
    }
if "ovb_summary" not in globals():
    ovb_mc = lab.mc_ovb(**ovb_params, reps=mc_reps, seed=1)
    ovb_summary = lab.summarize_estimates(ovb_mc, truth=ovb_params["beta_x"])

if "measurement_params" not in globals():
    measurement_params = {
        "n": 400,
        "beta_x": 1.0,
        "instrument_strength": 0.9,
        "latent_noise_sd": 1.0,
        "measurement_noise_sd": 1.1,
    }
if "me_summary" not in globals():
    me_mc = lab.mc_measurement_error(**measurement_params, reps=mc_reps, seed=2)
    me_summary = lab.summarize_estimates(me_mc, truth=measurement_params["beta_x"])

if "simultaneity_params" not in globals():
    simultaneity_params = {
        "n": 400,
        "demand_intercept": 10.0,
        "demand_slope": -0.8,
        "supply_intercept": 2.0,
        "supply_slope": 1.2,
        "instrument_strength": 1.5,
        "demand_shock_sd": 1.0,
        "supply_shock_sd": 1.0,
    }
if "sim_summary" not in globals():
    sim_mc = lab.mc_simultaneity(**simultaneity_params, reps=mc_reps, seed=3)
    sim_summary = lab.summarize_estimates(sim_mc, truth=simultaneity_params["demand_slope"])

comparison = pd.DataFrame(
    [
        {
            "Source": "Omitted variable bias",
            "Biased OLS mean": ovb_summary.loc[ovb_summary["Estimator"] == "OLS omit W", "Mean"].iloc[0],
            "Cleaner estimator mean": ovb_summary.loc[ovb_summary["Estimator"] == "IV using Z", "Mean"].iloc[0],
            "Truth": ovb_params["beta_x"],
        },
        {
            "Source": "Measurement error",
            "Biased OLS mean": me_summary.loc[me_summary["Estimator"] == "OLS using noisy X", "Mean"].iloc[0],
            "Cleaner estimator mean": me_summary.loc[me_summary["Estimator"] == "IV using Z", "Mean"].iloc[0],
            "Truth": measurement_params["beta_x"],
        },
        {
            "Source": "Simultaneity",
            "Biased OLS mean": sim_summary.loc[sim_summary["Estimator"] == "OLS of Q on P", "Mean"].iloc[0],
            "Cleaner estimator mean": sim_summary.loc[sim_summary["Estimator"] == "IV using supply shifter Z", "Mean"].iloc[0],
            "Truth": simultaneity_params["demand_slope"],
        },
    ]
)

display(comparison)

## Final takeaways

1. Different economic stories can produce the same econometric symptom: `cov(X, u) != 0`
2. Omitted variables can sometimes be fixed by adding the missing control, if it is observed
3. Measurement error and simultaneity usually need a different source of variation
4. That different source of variation is exactly what the IV lecture is about

If you want a clean classroom exercise, rerun each section with the bias switched off:

- OVB: set `rho_xw = 0`
- Measurement error: set `measurement_noise_sd = 0`
- Simultaneity: set `demand_shock_sd = 0` and inspect how the equilibrium cloud changes